<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/06-complex-nested-data/03-json-in-a-column.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 6.3 — JSON in a column: from_json, to_json, schema_of_json

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("6.3-json-in-a-column")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

# the classic shape: structured table with one JSON blob column
raw = spark.createDataFrame(
    [
        (1, '{"device": "ios",     "version": "3.2", "beta": true,  "screen": {"w": 390, "h": 844}}'),
        (2, '{"device": "web",     "version": "3.1", "locale": "de-DE"}'),
        (3, '{"device": "android", "version": "3.2", "beta": false, "screen": {"w": 412, "h": 915}}'),
        (4, 'oops not json'),
        (5, None),
    ],
    "user_id INT, props STRING",
)
raw.show(truncate=55)

## schema_of_json: draft the contract from a rich sample (then edit by hand)

In [ ]:
sample = raw.filter(col("user_id") == 1).first()["props"]
spark.range(1).select(F.schema_of_json(F.lit(sample)).alias("draft")).show(truncate=False)
# note what the draft CANNOT know: row 2's 'locale' (absent from the sample)

## from_json with the human-edited contract

In [ ]:
# drafted by schema_of_json, then edited: added locale, nested screen kept
PROPS = "device STRING, version STRING, beta BOOLEAN, locale STRING, screen STRUCT<w: INT, h: INT>"

parsed = raw.withColumn("p", F.from_json("props", PROPS))
parsed.select("user_id", "p.device", "p.locale", "p.screen.w", "p").show(truncate=40)
# absent fields -> null; unparseable (row 4) -> whole struct null; null in -> null out

In [ ]:
# The quarantine flag: distinguish 'bad JSON' from 'no JSON'
flagged = parsed.withColumn("_bad_json", col("props").isNotNull() & col("p").isNull())
flagged.select("user_id", "_bad_json").show()

## Surgical tools: get_json_object / json_tuple (strings only!)

In [ ]:
raw.select(
    "user_id",
    F.get_json_object("props", "$.device").alias("device"),
    F.get_json_object("props", "$.screen.w").alias("width_str"),   # STRING '390', not int
).show()

raw.select("user_id", F.json_tuple("props", "device", "version").alias("device", "version")).show()

## to_json: the way back out (Kafka/JDBC/webhook boundaries)

In [ ]:
outbound = parsed.filter(col("p").isNotNull()).select(
    "user_id",
    F.to_json(F.struct(
        "user_id",
        col("p.device").alias("device"),
        F.coalesce(col("p.beta"), F.lit(False)).alias("beta"),
    )).alias("payload"),
)
outbound.show(truncate=False)

## Parsing straight to arrays and maps

In [ ]:
arr = spark.createDataFrame([('[{"sku":"A1","qty":2},{"sku":"B2","qty":1}]',)], "items_json STRING")
arr.select(F.from_json("items_json", "ARRAY<STRUCT<sku: STRING, qty: INT>>").alias("items")) \
   .select(F.explode("items")).show()

# keys-unknown escape hatch: parse to a map (6.2 trade-offs apply)
raw.select(F.from_json("props", "MAP<STRING,STRING>").alias("m")) \
   .select(F.map_keys("m")).show(truncate=False)

## Exercises

1. FAILFAST edition: pass `options={"mode": "FAILFAST"}` to `from_json` and process row 4. When would you actually want this over the quarantine flag?
2. The repair pipeline: parse, set `beta=false` where null (`withField`, 6.1), drop `screen`, re-serialize with `to_json`. One chain.
3. Prove the parse-once advantage: extract device+version+locale via three `get_json_object` calls vs one `from_json` — compare the plans. How many times does each parse the text?
4. events.jsonl epilogue: read it as `spark.read.text`, then use `from_json` with the 6.0 `EVENTS_SCHEMA` to parse the `value` column — recreating the file reader by hand, corrupt line handled by the null pattern. Which lesson-5.5 pipeline did you just rebuild?

In [ ]:
# your exercise code here
